# Zadanie 1

In [37]:
import pandas as pd
from evidently.legacy.pipeline.column_mapping import ColumnMapping
from requests.packages import target
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier



#Generowanie zbioru historycznego
X_train, y_train = make_classification(n_samples=1000, n_features=5, random_state=42)
df_train = pd.DataFrame(X_train, columns=[f"feature_{i}" for i in range(5)])
df_train["target"] = y_train

#Generowanie zbioru produkcyjnego
X_prod, y_prod = make_classification(n_samples=300, n_features=5, random_state=999)
df_prod = pd.DataFrame(X_prod, columns=[f"feature_{i}" for i in range(5)])
df_prod["target"] = y_prod

#Podgląd danych
print("Zbior trening")
print(df_train.head())
print("\nZbior produk")
print(df_prod.head())
print("-" * 50)


model = RandomForestClassifier(random_state=42)
model.fit(df_train.drop("target", axis=1), df_train["target"])

df_train["prediction"] = model.predict(df_train.drop("target", axis=1))
df_prod["prediction"] = model.predict(df_prod.drop("target", axis=1))


print(f"Krztałt trening {df_train.shape}")
print("\nInfo trening")
print(df_train.info())
print("\nStatystyki trening")
print(df_train.describe())

print(f"Krztałt produk {df_prod.shape}")
print("\nInfo produk")
print(df_prod.info())
print("\nStatystyki produk")
print(df_prod.describe())



Zbior trening
   feature_0  feature_1  feature_2  feature_3  feature_4  target
0  -0.439643   0.542547  -0.822420   0.401366  -0.854840       0
1   2.822231  -2.480859  -1.147691  -2.101131   3.040278       1
2   1.618386  -1.369478  -2.084113  -1.179659   1.613602       1
3   1.659048  -0.615202   1.112688  -0.835098  -0.272205       1
4   1.849824  -1.679456  -0.926698  -1.402509   2.123129       1

Zbior produk
   feature_0  feature_1  feature_2  feature_3  feature_4  target
0   2.222586  -0.790005   2.018433   2.069610  -0.989938       0
1   2.279468  -0.664316   1.913283  -1.399254  -1.134046       0
2   1.211192  -0.375896   1.041245  -1.432447  -0.583922       0
3  -0.206273   1.431013  -1.646434   0.033596  -1.013327       0
4  -1.890763   0.422601  -1.448996  -0.075198   1.045211       1
--------------------------------------------------
Krztałt trening (1000, 7)

Info trening
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Co

# Zadanie 2

In [38]:
from evidently.presets import DataDriftPreset
from evidently import Report, DataDefinition, BinaryClassification, Dataset

df_prod_features = df_prod.drop(columns=['prediction'], errors='ignore')

data_drift_report = Report(metrics=[
    DataDriftPreset()
])

snapshot = data_drift_report.run(reference_data=df_train, current_data=df_prod_features)
snapshot.save_html("data_drift_report.html")

# Zadanie 3

In [39]:
from evidently.presets import ClassificationPreset
from evidently import Report
from evidently import DataDefinition
from evidently.core.datasets import BinaryClassification

data_def = DataDefinition(
    classification=[
        BinaryClassification(
            target="target",
            prediction_labels="prediction"
        )
    ]
)

reference_dataset = Dataset.from_pandas(df_train, data_def)
current_dataset = Dataset.from_pandas(df_prod, data_def)

performance_report = Report(metrics=[ClassificationPreset()])
performance_report.run(reference_dataset, current_dataset)
snapshot_p = performance_report.run(current_dataset, reference_dataset)
snapshot_p.save_html("performance_report.html")